# Automated Detection of Olive Fly Using Color-Space Segmentation, Morphological Feature Extraction, and a Multilayer Perceptron Neural Network
### Aleyna Musa, Ege Ramadanov, Georgi Markov

## Abstract

The olive fly (*Bactrocera oleae*) is one of the most economically damaging pests to olive cultivation worldwide. This notebook presents an automated image-based detection system designed to classify trap images as containing or not containing an olive fly. The system operates through three sequential stages: foreground segmentation using Otsu thresholding in the CIELAB color space, extraction of 13 handcrafted morphological and color features, and binary classification using a Multilayer Perceptron (MLP) neural network. To address class imbalance in the training dataset, image augmentation was applied to the minority class. A decision threshold was tuned by maximizing the F1 score. The final system achieved an accuracy of 89.83%, a precision of 60.61%, and a recall of 68.89% on the training dataset.

## 1. Introduction

The olive fly (*Bactrocera oleae*) is a major agricultural pest whose larvae feed on olive fruit, causing significant losses in yield and oil quality (Daane & Johnson, 2010). Early and accurate detection of olive flies in monitoring traps is essential for timely pest management intervention.

Traditional trap monitoring is performed manually, which is labor-intensive and prone to human error. The task is framed as a supervised binary classification problem with two classes: `olive_fly` (positive) and `not_olive_fly` (negative).

## 2. Dataset

The dataset consists of `.JPG` images organized into two directories:

- **`olive_fly/`** — images containing at least one olive fly specimen (630 images)
- **`not_olive_fly/`** — images of trap contents without olive flies (4,070 images)

This represents a class imbalance ratio of approximately 1:6.5 in favor of the negative class. Left unaddressed, this imbalance would bias the classifier toward always predicting the majority class. Section 4 describes how this was addressed through data augmentation.

## 3. Foreground Segmentation

Before features can be extracted, the insect or object of interest must be separated from the trap background. Standard sticky olive fly traps use a saturated yellow adhesive background, which is exploited for segmentation.

**Color Space Conversion:** Images are converted from BGR to the CIELAB color space (Commission Internationale de l'Éclairage, 1978). The b* channel represents the blue–yellow axis. Because the yellow trap background produces high b* values while dark foreground objects (insects, debris) produce low b* values, thresholding on b* separates the foreground more cleanly than a grayscale threshold, which is susceptible to lighting variation and shadow (Bradski & Kaehler, 2008).

**Otsu's Thresholding:** Rather than setting a fixed threshold manually, Otsu's method is applied automatically to the b* channel (Otsu, 1979). This algorithm selects the threshold that minimizes intra-class variance between the foreground and background pixel groups.

**Morphological Cleaning:** Two morphological operations clean the binary mask (Gonzalez & Woods, 2018): morphological opening (erosion followed by dilation) removes small noise pixels, and morphological closing (dilation followed by erosion) fills small holes within larger foreground regions. Both use a 7×7 rectangular kernel via `cv2.morphologyEx`.

**Connected Component Analysis:** The cleaned mask may still contain multiple disconnected regions. `skimage.measure.label` assigns a unique integer label to each connected region, and the largest region by pixel area is retained as the final foreground mask.

In [12]:
import os
import glob
import cv2
import numpy as np
from skimage.measure import label
from sklearn.neural_network import MLPClassifier

In [ ]:

def extract_foreground_mask(img, kernel_size=7):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    b_channel = lab[:, :, 2]
    _, bw = cv2.threshold(b_channel, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    labels = label(bw)
    if bw.sum() == 0:
        return np.zeros(bw.shape, np.uint8), labels
    largest = np.argmax(np.bincount(labels.flat, weights=bw.flat))
    return (labels == largest).astype(np.uint8), labels


def augment(img, rng):
    out = img.copy()
    if rng.rand() < 0.5: out = cv2.flip(out, 1)
    if rng.rand() < 0.5: out = cv2.flip(out, 0)
    M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2),
                                rng.uniform(-30, 30), rng.uniform(0.9, 1.1))
    out = cv2.warpAffine(out, M, (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REFLECT)
    out = np.clip(rng.uniform(0.85, 1.15) * out.astype(np.float32) + rng.uniform(-25, 25),
                  0, 255).astype(np.uint8)
    return out

## 4. Data Augmentation

To compensate for the class imbalance described in Section 2, synthetic positive samples are generated from the existing `olive_fly` images through random image augmentation until the positive class matches the size of the negative class (Shorten & Khoshgoftaar, 2019). The `augment` function applies the following operations:

- **Horizontal flip** (applied with 50% probability)
- **Vertical flip** (applied with 50% probability)
- **Random rotation** between −30° and +30° with a random scaling factor in [0.9, 1.1], using reflected border padding via `cv2.warpAffine`
- **Brightness and contrast jitter**: pixel values multiplied by a factor in [0.85, 1.15] and shifted by an offset in [−25, +25], then clipped to the valid [0, 255] range

All random operations use a fixed seed (42) for reproducibility (Harris et al., 2020).

In [14]:
# 13 features: shape (area_frac, aspect, solidity, circularity, eccentricity,
# hu1, n_components) + colour (mean H/S/V, Lab a*/b* under mask, fg-bg contrast)
def compute_features(image):
    H, W = image.shape[:2]
    mask, labels = extract_foreground_mask(image)
    area = float(mask.sum())
    area_frac = area / (H * W)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    aspect = solidity = circ = ecc = hu1 = 0.0
    if contours:
        c = max(contours, key=cv2.contourArea)
        ca = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        aspect = float(w) / h if h > 0 else 1.0
        hull = cv2.contourArea(cv2.convexHull(c))
        solidity = ca / hull if hull > 0 else 0.0
        perim = cv2.arcLength(c, True)
        circ = min(4 * np.pi * ca / (perim * perim), 1.5) if perim > 0 else 0.0
        if len(c) >= 5:
            (_, _), (MA, ma), _ = cv2.fitEllipse(c)
            ecc = min(max(MA, ma) / min(MA, ma), 5.0) if min(MA, ma) > 0 else 0.0
        hu = cv2.HuMoments(cv2.moments(c)).flatten()
        hu1 = float(-np.sign(hu[0]) * np.log10(abs(hu[0]) + 1e-30))

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    m = mask.astype(bool)
    if m.any():
        hue = float(hsv[:, :, 0][m].mean())
        sat = float(hsv[:, :, 1][m].mean())
        val = float(hsv[:, :, 2][m].mean())
        lab_a = float(lab[:, :, 1][m].mean())
        lab_b = float(lab[:, :, 2][m].mean())
        bg = gray[~m].mean() if (~m).any() else gray[m].mean()
        contrast = float(bg - gray[m].mean())
    else:
        hue = sat = val = lab_a = lab_b = contrast = 0.0

    counts = np.bincount(labels.flat)
    if len(counts): counts[0] = 0
    ncomp = float((counts > 0.1 * area).sum()) if area > 0 else 0.0

    return np.array([area_frac, aspect, solidity, circ, ecc,
                     hue, sat, val, lab_a, lab_b, ncomp, contrast, hu1])

## 5. Feature Extraction

For each image, 13 numerical features are extracted from the segmented foreground mask, grouped into shape and color descriptors.

### Shape Features

| # | Feature | Description |
|---|---------|-------------|
| 1 | `area_frac` | Foreground pixels / total image pixels — relative object size |
| 2 | `aspect` | Width / height of the bounding rectangle |
| 3 | `solidity` | Contour area / convex hull area — measures compactness |
| 4 | `circularity` | 4π·area / perimeter² — closeness to a circle (clipped at 1.5) |
| 5 | `eccentricity` | Major axis / minor axis of fitted ellipse — elongation (clipped at 5.0) |
| 6 | `hu1` | First Hu moment (Hu, 1962), log-transformed: −sign(h₁)·log₁₀(\|h₁\|) — rotation and scale invariant shape descriptor |
| 7 | `ncomp` | Number of connected components whose area exceeds 10% of total foreground |

### Color Features (computed only over foreground mask pixels)

| # | Feature | Description |
|---|---------|-------------|
| 8 | `hue` | Mean HSV hue of foreground pixels |
| 9 | `sat` | Mean HSV saturation of foreground pixels |
| 10 | `val` | Mean HSV brightness of foreground pixels |
| 11 | `lab_a` | Mean CIELAB a* (green–red axis) of foreground pixels |
| 12 | `lab_b` | Mean CIELAB b* (blue–yellow axis) of foreground pixels |
| 13 | `contrast` | Mean background gray intensity − mean foreground gray intensity |

## 6. Feature Standardization and Neural Network Training

Raw feature values span very different numerical ranges (e.g., `area_frac` is in [0, 1] while `hue` spans [0, 180]). Training a neural network on unstandardized features causes large-magnitude features to dominate gradient updates, slowing or destabilizing convergence (LeCun et al., 2002). Each feature is therefore z-score standardized:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

where μ and σ are the per-feature mean and standard deviation computed from the full balanced training set.

A **Multilayer Perceptron (MLP)** is used for binary classification (Rumelhart et al., 1986) with the architecture **13 → 10 → 1**:

- **Hidden layer:** 10 neurons with ReLU activation — f(x) = max(0, x) (Nair & Hinton, 2010)
- **Output layer:** 1 neuron with sigmoid activation — p = 1 / (1 + e^−z), producing a probability in [0, 1]

The model is trained using scikit-learn's `MLPClassifier` (Pedregosa et al., 2011) with 8,000 maximum iterations. The **decision threshold** is tuned over 81 values in [0.1, 0.9] on the real non-augmented images to maximize the F1 score (Fawcett, 2006). The optimal threshold was **0.70**.

In [ ]:
rng = np.random.RandomState(42)

print("Reading data...")
pos = [cv2.imread(f) for f in (glob.glob("olive_fly/*.JPG") + glob.glob("olive_fly/*.jpg"))]
neg = [cv2.imread(f) for f in (glob.glob("not_olive_fly/*.JPG") + glob.glob("not_olive_fly/*.jpg"))]
pos = [i for i in pos if i is not None]
neg = [i for i in neg if i is not None]
print(f"Class counts before balancing: olive_fly={len(pos)}, not_olive_fly={len(neg)}")

aug = [augment(pos[rng.randint(len(pos))], rng) for _ in range(len(neg) - len(pos))]
feats = lambda imgs: np.array([compute_features(i) for i in imgs])
X = np.vstack([feats(pos), feats(aug), feats(neg)])
y = np.array([1] * (len(pos) + len(aug)) + [0] * len(neg))
print(f"Class counts after balancing: olive_fly={int((y==1).sum())}, not_olive_fly={int((y==0).sum())}")

X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std[X_std == 0] = 1e-8
X_scaled = (X - X_mean) / X_std

mlp = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', max_iter=8000, random_state=42)
mlp.fit(X_scaled, y)

Xreal = np.vstack([feats(pos), feats(neg)])
yreal = np.array([1] * len(pos) + [0] * len(neg))
proba = mlp.predict_proba((Xreal - X_mean) / X_std)[:, 1]
def f1_at(t):
    pred = proba >= t
    tp = int((pred & (yreal == 1)).sum()); fp = int((pred & (yreal == 0)).sum()); fn = int((~pred & (yreal == 1)).sum())
    if tp == 0: return 0.0
    p = tp / (tp + fp); r = tp / (tp + fn); return 2 * p * r / (p + r)
THRESHOLD = float(max(np.linspace(0.1, 0.9, 81), key=f1_at))
print(f"Tuned THRESHOLD = {THRESHOLD:.2f}")

print(f"X_MEAN = np.array({list(X_mean)})")
print(f"X_STD  = np.array({list(X_std)})")
print(f"W1 = np.array({mlp.coefs_[0].tolist()})")
print(f"B1 = np.array({mlp.intercepts_[0].tolist()})")
print(f"W2 = np.array({mlp.coefs_[1].tolist()})")
print(f"B2 = np.array({mlp.intercepts_[1].tolist()})")

Reading data...
Class counts before balancing: olive_fly=630, not_olive_fly=4070
Class counts after balancing: olive_fly=4070, not_olive_fly=4070
Tuned THRESHOLD = 0.70
X_MEAN = np.array([np.float64(0.08433049658655181), np.float64(1.1022629016259757), np.float64(0.7888162836470152), np.float64(0.474876311838575), np.float64(1.9724634663899927), np.float64(36.7801783205339), np.float64(123.79012454398972), np.float64(91.32068834980697), np.float64(130.80685660087326), np.float64(143.14316761762797), np.float64(2.862776412776413), np.float64(47.30772004759707), np.float64(0.5975299443079309)])
X_STD  = np.array([np.float64(0.07419576975967647), np.float64(0.55175112499908), np.float64(0.13219707493637348), np.float64(0.17401921165867676), np.float64(0.7818849445614432), np.float64(28.700886739431624), np.float64(50.500637145919725), np.float64(47.25658267194249), np.float64(4.619529891414145), np.float64(12.594588411262658), np.float64(2.2313304405164334), np.float64(31.096603766238175)

## 7. Inference Pipeline

The trained model weights and normalization constants (`X_MEAN`, `X_STD`, `W1`, `B1`, `W2`, `B2`) are extracted and stored below. This makes the inference pipeline fully self-contained — no scikit-learn or saved model file is needed at runtime, which is important for deployment on resource-constrained hardware such as a Raspberry Pi. The `detect_olive_fly` function replicates the MLP forward pass manually using only NumPy and OpenCV.

In [16]:
import glob
import cv2
import numpy as np
from skimage.measure import label

In [17]:
MODEL_PARAMS = {
    'X_MEAN': X_mean,
    'X_STD': X_std,
    'W1': mlp.coefs_[0],
    'B1': mlp.intercepts_[0],
    'W2': mlp.coefs_[1],
    'B2': mlp.intercepts_[1]
}

X_MEAN = MODEL_PARAMS['X_MEAN']
X_STD = MODEL_PARAMS['X_STD']

W1 = MODEL_PARAMS['W1']
B1 = MODEL_PARAMS['B1']

W2 = MODEL_PARAMS['W2']
B2 = MODEL_PARAMS['B2']


In [ ]:

def extract_foreground_mask(img, kernel_size=7):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    b_channel = lab[:, :, 2]
    _, bw = cv2.threshold(b_channel, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    labels = label(bw)
    if bw.sum() == 0:
        return np.zeros(bw.shape, np.uint8), labels
    largest = np.argmax(np.bincount(labels.flat, weights=bw.flat))
    return (labels == largest).astype(np.uint8), labels

# 13 features: shape (area_frac, aspect, solidity, circularity, eccentricity,
# hu1, n_components) + colour (mean H/S/V, Lab a*/b* under mask, fg-bg contrast)
def compute_features(image):
    H, W = image.shape[:2]
    mask, labels = extract_foreground_mask(image)
    area = float(mask.sum())
    area_frac = area / (H * W)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    aspect = solidity = circ = ecc = hu1 = 0.0
    if contours:
        c = max(contours, key=cv2.contourArea)
        ca = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        aspect = float(w) / h if h > 0 else 1.0
        hull = cv2.contourArea(cv2.convexHull(c))
        solidity = ca / hull if hull > 0 else 0.0
        perim = cv2.arcLength(c, True)
        circ = min(4 * np.pi * ca / (perim * perim), 1.5) if perim > 0 else 0.0
        if len(c) >= 5:
            (_, _), (MA, ma), _ = cv2.fitEllipse(c)
            ecc = min(max(MA, ma) / min(MA, ma), 5.0) if min(MA, ma) > 0 else 0.0
        hu = cv2.HuMoments(cv2.moments(c)).flatten()
        hu1 = float(-np.sign(hu[0]) * np.log10(abs(hu[0]) + 1e-30))

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    m = mask.astype(bool)
    if m.any():
        hue = float(hsv[:, :, 0][m].mean())
        sat = float(hsv[:, :, 1][m].mean())
        val = float(hsv[:, :, 2][m].mean())
        lab_a = float(lab[:, :, 1][m].mean())
        lab_b = float(lab[:, :, 2][m].mean())
        bg = gray[~m].mean() if (~m).any() else gray[m].mean()
        contrast = float(bg - gray[m].mean())
    else:
        hue = sat = val = lab_a = lab_b = contrast = 0.0

    counts = np.bincount(labels.flat)
    if len(counts): counts[0] = 0
    ncomp = float((counts > 0.1 * area).sum()) if area > 0 else 0.0

    return np.array([area_frac, aspect, solidity, circ, ecc,
                     hue, sat, val, lab_a, lab_b, ncomp, contrast, hu1])

In [19]:

def detect_olive_fly(image) -> bool:

    try:
        # 1. Extract and standardize physical attributes
        raw_features = compute_features(image)
        scaled_features = (raw_features - X_MEAN) / X_STD

        # 2. Hidden Layer Propagation: Z1 = A1*W1 + B1
        z1 = np.dot(scaled_features, W1) + B1
        a2 = np.maximum(0, z1)  # ReLU non-linear activation function

        # 3. Output Layer Propagation: Z2 = A2*W2 + B2
        z2 = np.dot(a2, W2) + B2

        # Sigmoid activation function mapping to classification boundaries
        probability = 1 / (1 + np.exp(-np.clip(z2[0], -500, 500)))

        return bool(probability >= THRESHOLD)

    except Exception:
        return False

## 8. Results and Evaluation

The classifier is evaluated on the full labeled training dataset (excluding augmented samples) using the tuned threshold of 0.70. The confusion matrix and metrics are computed below.

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP = 217 | FN = 98 |
| **Actual Negative** | FP = 141 | TN = 1,894 |

- **Accuracy** = (217 + 1,894) / 2,350 = **89.83%**
- **Precision** = 217 / (217 + 141) = **60.61%**
- **Recall** = 217 / (217 + 98) = **68.89%**

In [20]:
def evaluate_classifier(directory):
    TP, TN, FP, FN = 0, 0, 0, 0
    
    for filename in glob.glob(str(directory)+"/**/*.JPG", recursive=True):
        img = cv2.imread(filename)
        if "not_olive_fly" in filename:
            olive_fly = False
        elif "olive_fly" in filename:
            olive_fly = True
        else:
            print(f"{filename} not labeled.")
            continue

        detection_result = detect_olive_fly(img)

        if olive_fly and detection_result:
            TP += 1
        elif olive_fly and not detection_result:
            FN += 1
        elif not olive_fly and detection_result:
            FP += 1
        else:
            TN += 1
            
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    if (TP + FP) > 0 and (TP + FN) > 0:
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        accuracy = (TP + TN) / (TP + TN + FP + FN)
        print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, Accuracy: {accuracy:.4f}")

In [21]:
print("Evaluating on training data...\n")
evaluate_classifier(".")

Evaluating on training data...

TP: 217, TN: 1894, FP: 141, FN: 98
Precision: 0.6061, Recall: 0.6889, Accuracy: 0.8983


## 9. Conclusion

This notebook presented an automated olive fly detection pipeline designed for deployment on resource-constrained embedded hardware. The system combines LAB color-space segmentation, morphological image processing, and a compact 13-feature MLP classifier. At inference time the model is implemented entirely using OpenCV and NumPy, eliminating heavy dependencies such as scikit-learn or deep learning frameworks. The system achieves 89.83% accuracy on the training dataset with an F1-optimized threshold of 0.70.

---

## References

<p style="padding-left: 2em; text-indent: -2em;">Bradski, G. (2000). The OpenCV library. <em>Dr. Dobb's Journal of Software Tools</em>, <em>25</em>(11), 120–125.</p>

<p style="padding-left: 2em; text-indent: -2em;">Bradski, G., & Kaehler, A. (2008). <em>Learning OpenCV: Computer vision with the OpenCV library</em>. O'Reilly Media.</p>

<p style="padding-left: 2em; text-indent: -2em;">Commission Internationale de l'Éclairage. (1978). <em>Recommendations on uniform color spaces, color-difference equations, psychometric color terms</em> (CIE Publication No. 15). Bureau Central de la CIE.</p>

<p style="padding-left: 2em; text-indent: -2em;">Daane, K. M., & Johnson, M. W. (2010). Olive fruit fly: Managing an ancient pest in modern times. <em>Annual Review of Entomology</em>, <em>55</em>, 291–310. https://doi.org/10.1146/annurev.ento.54.110807.090553</p>

<p style="padding-left: 2em; text-indent: -2em;">Fawcett, T. (2006). An introduction to ROC analysis. <em>Pattern Recognition Letters</em>, <em>27</em>(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010</p>

<p style="padding-left: 2em; text-indent: -2em;">Gonzalez, R. C., & Woods, R. E. (2018). <em>Digital image processing</em> (4th ed.). Pearson.</p>

<p style="padding-left: 2em; text-indent: -2em;">Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. <em>Nature</em>, <em>585</em>(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2</p>

<p style="padding-left: 2em; text-indent: -2em;">Hu, M.-K. (1962). Visual pattern recognition by moment invariants. <em>IRE Transactions on Information Theory</em>, <em>8</em>(2), 179–187. https://doi.org/10.1109/TIT.1962.1057692</p>

<p style="padding-left: 2em; text-indent: -2em;">LeCun, Y., Bottou, L., Orr, G. B., & Müller, K.-R. (2002). Efficient backprop. In G. B. Orr & K.-R. Müller (Eds.), <em>Neural networks: Tricks of the trade</em> (pp. 9–50). Springer. https://doi.org/10.1007/3-540-49430-8_2</p>

<p style="padding-left: 2em; text-indent: -2em;">Nair, V., & Hinton, G. E. (2010). Rectified linear units improve restricted Boltzmann machines. In J. Fürnkranz & T. Joachims (Eds.), <em>Proceedings of the 27th International Conference on Machine Learning</em> (pp. 807–814). Omnipress.</p>

<p style="padding-left: 2em; text-indent: -2em;">Otsu, N. (1979). A threshold selection method from gray-level histograms. <em>IEEE Transactions on Systems, Man, and Cybernetics</em>, <em>9</em>(1), 62–66. https://doi.org/10.1109/TSMC.1979.4310076</p>

<p style="padding-left: 2em; text-indent: -2em;">Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. <em>Journal of Machine Learning Research</em>, <em>12</em>, 2825–2830.</p>

<p style="padding-left: 2em; text-indent: -2em;">Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. <em>Nature</em>, <em>323</em>(6088), 533–536. https://doi.org/10.1038/323533a0</p>

<p style="padding-left: 2em; text-indent: -2em;">Shorten, C., & Khoshgoftaar, T. M. (2019). A survey on image data augmentation for deep learning. <em>Journal of Big Data</em>, <em>6</em>(1), Article 60. https://doi.org/10.1186/s40537-019-0197-0</p>

<p style="padding-left: 2em; text-indent: -2em;">van der Walt, S., Schönberger, J. L., Nunez-Iglesias, J., Boulogne, F., Warner, J. D., Yager, N., Gouillart, E., & Yu, T. (2014). scikit-image: Image processing in Python. <em>PeerJ</em>, <em>2</em>, e453. https://doi.org/10.7717/peerj.453</p>

In [22]:
test_image_path = "olive_fly/IMG_0597 1 referencia.JPG"


if os.path.exists(test_image_path):
    test_img = cv2.imread(test_image_path)
    if test_img is not None:
        result = detect_olive_fly(test_img)
        print(f"Image: {test_image_path}")
        print(f"Detection Result: {'Olive Fly Detected' if result else 'No Olive Fly'}")
    else:
        print(f"Could not read image: {test_image_path}")
else:
    print(f"File not found: {test_image_path}")

Image: olive_fly/IMG_0597 1 referencia.JPG
Detection Result: Olive Fly Detected
